# Worked Capstone: Text Sentiment Classification and Error Analysis

**Domain:** Natural language processing  
**Primary dataset:** `text_reviews.csv`  
**Level:** Practitioner to Advanced

## Business goal

Build a transparent TF-IDF sentiment baseline and use errors and coefficients to understand representation limits before deep learning.

This is a worked reference project. First attempt the corresponding phase project independently; then use this capstone to compare framing, evaluation, code structure, and communication.

## Decision questions

        1. What does the vocabulary learn?
2. How strong is a linear text baseline?
3. Which tokens drive predictions?
4. Which errors reveal dataset/model limits?

        ## Definition of done

        - [ ] Text split
- [ ] TF-IDF pipeline
- [ ] Cross-validation
- [ ] Holdout metrics
- [ ] Top terms
- [ ] Error examples
- [ ] Artifact

## End-to-end workflow

```text
Decision and scope
      ↓
Data contract and quality
      ↓
Exploration and hypotheses
      ↓
Baseline and evaluation design
      ↓
Candidate method(s)
      ↓
Held-out / temporal evaluation
      ↓
Error, slice, and sensitivity analysis
      ↓
Artifacts, limitations, recommendation
```

At every stage, distinguish calculation correctness, statistical validity, operational validity, and decision validity.

## Risk register

        | Risk | Mitigation |
        |---|---|
        | Duplicate templates cross splits | Audit near-duplicates in real datasets. |
| Token importance treated as semantic truth | Interpret coefficients within representation and data. |
| Synthetic performance overgeneralized | Limit scope and use real-domain validation. |

In [ ]:
from pathlib import Path
import sys
import json
import warnings
warnings.filterwarnings("ignore")

_candidates = [Path.cwd(), *Path.cwd().parents]
COURSE_ROOT = next((p for p in _candidates if (p / "datasets").exists()), Path.cwd())
DATA_DIR = COURSE_ROOT / "datasets"
ARTIFACT_DIR = COURSE_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(COURSE_ROOT))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f"Course root: {COURSE_ROOT}")

## 1. Load and split

Inspect class balance and reserve a stratified holdout.

In [ ]:
from sklearn.model_selection import train_test_split,StratifiedKFold,cross_validate
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,roc_auc_score,average_precision_score
import joblib

df=pd.read_csv(DATA_DIR/"text_reviews.csv")
Xdev,Xtest,ydev,ytest=train_test_split(df.review_text,df.positive,test_size=.25,stratify=df.positive,random_state=42)
print(df.positive.value_counts(normalize=True))
display(df.sample(5,random_state=42))

## 2. Pipeline and validation

The vectorizer must be fitted within each fold so validation vocabulary statistics remain unseen.

In [ ]:
pipeline=Pipeline([
    ("tfidf",TfidfVectorizer(ngram_range=(1,2),min_df=2,max_df=.98,sublinear_tf=True)),
    ("model",LogisticRegression(max_iter=1000)),
])
cv=StratifiedKFold(5,shuffle=True,random_state=42)
scores=cross_validate(pipeline,Xdev,ydev,cv=cv,scoring=["roc_auc","average_precision","accuracy"])
display(pd.DataFrame(scores)[["test_roc_auc","test_average_precision","test_accuracy"]].describe())

## 3. Holdout evaluation

Evaluate ranking and thresholded outputs.

In [ ]:
pipeline.fit(Xdev,ydev)
prob=pipeline.predict_proba(Xtest)[:,1]
pred=(prob>=.5).astype(int)
print("ROC-AUC:",roc_auc_score(ytest,prob),"PR-AUC:",average_precision_score(ytest,prob))
print(classification_report(ytest,pred,digits=3))

## 4. Coefficients and errors

Inspect model reliance and ambiguous/mistaken examples.

In [ ]:
terms=np.array(pipeline.named_steps["tfidf"].get_feature_names_out())
coef=pipeline.named_steps["model"].coef_[0]
top_positive=pd.Series(coef,index=terms).nlargest(12)
top_negative=pd.Series(coef,index=terms).nsmallest(12)
display(pd.DataFrame({"positive_term":top_positive.index,"positive_weight":top_positive.values}))
display(pd.DataFrame({"negative_term":top_negative.index,"negative_weight":top_negative.values}))

errors=pd.DataFrame({"text":Xtest.to_numpy(),"actual":ytest.to_numpy(),"probability":prob,"prediction":pred})
errors=errors[errors.actual!=errors.prediction].sort_values("probability")
display(errors.head(10))

## 5. Save artifact and extension

Persist the full text pipeline. A transformer extension should retain the same split, baseline, metrics, and error protocol.

In [ ]:
artifact=ARTIFACT_DIR/"capstone_text_sentiment_pipeline.joblib"
joblib.dump(pipeline,artifact)
print(artifact)
print("Advanced extension: fine-tune a pretrained encoder only after documenting domain, tokenization, compute, privacy, and serving constraints.")

## Model/project card

Complete this before presenting the result:

| Field | Statement |
|---|---|
| Intended use | |
| Excluded use | |
| Data population and coverage | |
| Target/metric definition | |
| Evaluation split | |
| Baseline | |
| Primary result | |
| Known limitations | |
| Important subgroup behaviour | |
| Human review / abstention | |
| Monitoring | |
| Owner and review cadence | |

## Final reflection

1. Which result changed your initial belief?
2. Which assumption creates the largest residual risk?
3. What simpler alternative was competitive?
4. What evidence is still required before an operational decision?
5. What would you monitor first after release?

Re-run the notebook from a clean kernel and verify generated artifacts before considering the capstone complete.